# 00 · Unificación PISA (2000–2025)

**Objetivo:** juntar en una sola tabla todos los ciclos de PISA, con las mismas columnas en todos los años.

| Años | Fuente | Archivo en `data/raw/` |
|---|---|---|
| 2000–2018 | Banco Mundial, EdStats (base 12), descargado en la sección 1 con `wbgapi` | `PISA_BancoMundial_2000_2018.csv` |
| 2022 | OCDE, *PISA 2022 Results, Volume I* (tablas web) | `PISA2022_cap*.xlsx` |
| 2025 | OCDE, *PISA 2025 Results, Volume I* (tablas web) | `PISA2025_cap*.xlsx` |

**Salida:** `data/processed/pisa_unificado.csv`: una fila por país y ciclo, con `ISO3_code` y `Time` para unir con WPP2024.

Las columnas son las del Banco Mundial, porque es la fuente más antigua. Se quedaron solo las que la OCDE también publica en 2022 y 2025: por eso no hay P05 ni P95.

In [1]:
%load_ext autoreload
%autoreload 2

import re
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # permite importar src/

import pandas as pd
import wbgapi as wb

from src.utils import DATA_RAW, cargar_raw, guardar_processed, resumen

pd.set_option("display.max_columns", None)

## 1. Banco Mundial (2000–2018)

Puntajes PISA de la base 12 (Education Statistics) del Banco Mundial: medias, percentiles y niveles de competencia, en total y por sexo, más el % con nivel mínimo según origen migratorio.

La descarga tarda unos minutos, así que solo se hace si el CSV no está en `data/raw/`. Para volver a descargar, borra el archivo o pon `FORZAR_DESCARGA = True`.

El Banco Mundial solo llega hasta 2018. 2022 y 2025 salen de la OCDE (sección 2).

In [2]:
FORZAR_DESCARGA = False
ARCHIVO_BANCO_MUNDIAL = "PISA_BancoMundial_2000_2018.csv"

# Materia -> (prefijo del código del Banco Mundial, nombre de columna)
MATERIAS = {
    "MAT": "Math",
    "REA": "Reading",
    "SCI": "Science",
}

# Sufijo del código del Banco Mundial -> sufijo de la columna (total, mujeres, hombres)
SEXOS = {"": "Total", ".FE": "Female", ".MA": "Male"}

# Solo los percentiles que también publica la OCDE en PISA 2022 y 2025 (sin P05 ni P95)
PERCENTILES = ["P10", "P25", "P50", "P75", "P90"]

# Niveles de competencia (% de estudiantes de 15 años) tal como los publica el Banco Mundial.
# Código del nivel -> etiqueta de columna. Cambian entre materias y entre ediciones de PISA.
NIVELES = {
    "MAT": {"0": "Below1", "1": "L1", "2": "L2", "3": "L3", "4": "L4", "5": "L5", "6": "L6"},
    "REA": {
        "0.B1C": "Below1C", "1C": "L1C", "1B": "L1B", "1A": "L1A",
        "2": "L2", "3": "L3", "4": "L4", "5": "L5", "6": "L6",
    },
    "SCI": {
        "0": "Below1B", "1B": "L1B", "1A": "L1A",
        "2": "L2", "3": "L3", "4": "L4", "5": "L5", "6": "L6",
    },
}

# % de estudiantes (fin de secundaria baja) con al menos el nivel mínimo de competencia, según
# antecedente migratorio. Son series de la UNESCO (UIS, ODS 4.1.1): no son puntajes medios y no
# se desglosan por sexo. En los años de PISA coinciden con la OCDE (ver sección 4).
MIGRACION = {
    "UIS.MATH.LOWERSEC.NATIVE": "PISA_Math_MinProf_NonImmigrant",
    "UIS.MATH.LOWERSEC.NONNATIVE": "PISA_Math_MinProf_Immigrant",
    "UIS.READ.LOWERSEC.NATIVE": "PISA_Reading_MinProf_NonImmigrant",
    "UIS.READ.LOWERSEC.NONNATIVE": "PISA_Reading_MinProf_Immigrant",
}

# Solo los años en que se aplicó PISA (evita timeouts de la API)
PISA_YEARS = [2000, 2003, 2006, 2009, 2012, 2015, 2018]

# Series por petición: con todas juntas la API tarda demasiado
TAMANO_LOTE = 12

In [3]:
def construir_indicadores():
    # Diccionario {código Banco Mundial: nombre de columna} con todas las series a descargar
    indicadores = {}
    for cod, nombre in MATERIAS.items():
        # Medias: total y por sexo
        for suf_cod, suf_nom in SEXOS.items():
            indicadores[f"LO.PISA.{cod}{suf_cod}"] = f"PISA_{nombre}_{suf_nom}"
        # Percentiles de la distribución (solo total)
        for p in PERCENTILES:
            indicadores[f"LO.PISA.{cod}.{p}"] = f"PISA_{nombre}_{p}"
        # Niveles de competencia: total y por sexo
        for nivel, etiqueta in NIVELES[cod].items():
            for suf_cod, suf_nom in SEXOS.items():
                indicadores[f"LO.PISA.{cod}.{nivel}{suf_cod}"] = f"PISA_{nombre}_{etiqueta}_{suf_nom}"
    indicadores.update(MIGRACION)
    return indicadores


def descargar_banco_mundial(indicadores):
    # Datos crudos en formato ancho: economy, series, YR2000, YR2003...
    codigos = list(indicadores)
    lotes = []
    for i in range(0, len(codigos), TAMANO_LOTE):
        lote = codigos[i : i + TAMANO_LOTE]
        print(f"Descargando series {i + 1}-{i + len(lote)} de {len(codigos)}...")
        lotes.append(wb.data.DataFrame(lote, time=PISA_YEARS, labels=False, db=12).reset_index())
    return pd.concat(lotes, ignore_index=True)


def ordenar_banco_mundial(crudo, indicadores):
    # Una fila por país/año y una columna por indicador
    largo = crudo.melt(id_vars=["economy", "series"], var_name="Time", value_name="Score")
    # Quitar el prefijo 'YR' para dejar el año como entero (compatible con WPP2024)
    largo["Time"] = largo["Time"].str.replace("YR", "").astype(int)

    ancho = largo.pivot_table(index=["economy", "Time"], columns="series", values="Score").reset_index()
    ancho.columns.name = None
    ancho = ancho.rename(columns={"economy": "ISO3_code", **indicadores})

    # PISA solo tiene registros en los años de aplicación
    return ancho.dropna(subset=["PISA_Math_Total"]).reset_index(drop=True)

In [4]:
INDICADORES = construir_indicadores()
ruta_bm = DATA_RAW / ARCHIVO_BANCO_MUNDIAL

if FORZAR_DESCARGA or not ruta_bm.exists():
    banco_mundial = ordenar_banco_mundial(descargar_banco_mundial(INDICADORES), INDICADORES)
    banco_mundial.to_csv(ruta_bm, index=False)
    print(f"Guardado: {ruta_bm.name}")
else:
    banco_mundial = cargar_raw(ARCHIVO_BANCO_MUNDIAL)

print(f"{len(INDICADORES)} series | {banco_mundial.shape}")
banco_mundial["Time"].value_counts().sort_index()

Descargando series 1-12 de 100...


Descargando series 13-24 de 100...


Descargando series 25-36 de 100...


Descargando series 37-48 de 100...


Descargando series 49-60 de 100...


Descargando series 61-72 de 100...


Descargando series 73-84 de 100...


Descargando series 85-96 de 100...


Descargando series 97-100 de 100...


Guardado: PISA_BancoMundial_2000_2018.csv
100 series | (419, 102)


Time
2000    42
2003    41
2006    56
2009    69
2012    62
2015    71
2018    78
Name: count, dtype: int64

## 2. Tablas de la OCDE (2022 y 2025)

Cada libro de Excel trae una tabla por hoja. Estas son las que se usan:

| Contenido | 2022 | 2025 |
|---|---|---|
| Media y percentiles | I.B1.2.1–2.3 | I.B1.2a.1–2a.3 |
| Niveles de competencia | I.B1.3.1–3.3 | I.B1.2a.10–2a.12 |
| Media por sexo | I.B1.4.17–4.19 | I.B1.2c.1–2c.3 |
| Niveles por sexo | I.B1.4.28–4.30 | I.B1.2c.10–2c.12 |
| Bajo rendimiento por origen migratorio | I.B1.7.37 (mat.), 7.41 (lect.) | I.B1.2d.15 (mat.), 2d.14 (lect.) |

En 2025 el orden de las tablas es ciencias, lectura y matemáticas, al revés que en 2022.

**Armonización con el Banco Mundial:**
- **Niveles de matemáticas:** desde 2022 la escala tiene Below 1c, 1c, 1b y 1a. En el Banco Mundial `Below1` es todo lo que queda por debajo de 357,77 puntos, así que `Below1 = Below 1c + 1c + 1b`, y `L1` corresponde al nivel 1a. Lectura y ciencias ya usaban las mismas categorías.
- **Origen migratorio:** la OCDE publica el % de estudiantes con bajo rendimiento (por debajo del nivel 2). La columna del Banco Mundial (UIS) es el % que alcanza al menos el nivel 2, así que `MinProf = 100 − % bajo rendimiento`.
- **Celdas `c` y `m`** (muestra insuficiente o dato no disponible) se leen como nulos.

In [5]:
FUENTES = {
    2022: {
        "media": ("PISA2022_cap2_desempeno.xlsx",
                  {"Math": "Table I.B1.2.1", "Reading": "Table I.B1.2.2", "Science": "Table I.B1.2.3"}),
        "niveles": ("PISA2022_cap3_niveles.xlsx",
                    {"Math": "Table I.B1.3.1", "Reading": "Table I.B1.3.2", "Science": "Table I.B1.3.3"}),
        "genero": ("PISA2022_cap4_equidad_genero.xlsx",
                   {"Math": "Table I.B1.4.17", "Reading": "Table I.B1.4.18", "Science": "Table I.B1.4.19"}),
        "niveles_genero": ("PISA2022_cap4_equidad_genero.xlsx",
                           {"Math": "Table I.B1.4.28", "Reading": "Table I.B1.4.29", "Science": "Table I.B1.4.30"}),
        "migracion": ("PISA2022_cap7_migracion.xlsx",
                      {"Math": "Table I.B1.7.37", "Reading": "Table I.B1.7.41"}),
    },
    2025: {
        "media": ("PISA2025_cap2a_desempeno_niveles.xlsx",
                  {"Math": "Table I.B1.2a.3", "Reading": "Table I.B1.2a.2", "Science": "Table I.B1.2a.1"}),
        "niveles": ("PISA2025_cap2a_desempeno_niveles.xlsx",
                    {"Math": "Table I.B1.2a.12", "Reading": "Table I.B1.2a.11", "Science": "Table I.B1.2a.10"}),
        "genero": ("PISA2025_cap2c_genero.xlsx",
                   {"Math": "Table I.B1.2c.3", "Reading": "Table I.B1.2c.2", "Science": "Table I.B1.2c.1"}),
        "niveles_genero": ("PISA2025_cap2c_genero.xlsx",
                           {"Math": "Table I.B1.2c.12", "Reading": "Table I.B1.2c.11", "Science": "Table I.B1.2c.10"}),
        "migracion": ("PISA2025_cap2d_migracion.xlsx",
                      {"Math": "Table I.B1.2d.15", "Reading": "Table I.B1.2d.14"}),
    },
}

# La tabla de migración de 2025 trae 2015, 2018, 2022 y 2025 lado a lado: se toma el bloque de 2025
PREFIJO_MIGRACION = {2022: (), 2025: ("PISA 2025",)}

PERCENTILES_OCDE = {"P10": "10th", "P25": "25th", "P50": "Median", "P75": "75th", "P90": "90th"}
SEXOS_OCDE = {"Female": "Girls", "Male": "Boys"}
MATERIA_COD = {nombre: cod for cod, nombre in MATERIAS.items()}  # "Math" -> "MAT"

### Lectura de las hojas

Todas las hojas tienen la misma forma: el título arriba, varias filas de encabezado con celdas combinadas y luego una fila por país. `leer_tabla` convierte cada columna en una tupla con su ruta de encabezados, por ejemplo `('Girls', 'Mean', 'Mean score')`, y descarta los errores estándar (`S.E.`).

In [6]:
def leer_tabla(archivo, hoja):
    df = pd.read_excel(DATA_RAW / archivo, sheet_name=hoja, header=None)
    valores = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")  # 'c' y 'm' -> NaN

    # Fila de datos: tiene nombre de país y al menos un número
    es_dato = df[0].notna() & valores.notna().any(axis=1)
    inicio = es_dato.idxmax()

    # Encabezados: las celdas combinadas solo tienen valor en la primera columna -> rellenar a la derecha
    encabezado = df.iloc[3:inicio, 1:].ffill(axis=1)
    etiquetas = [
        tuple(" ".join(str(v).split()) for v in encabezado[c] if pd.notna(v))
        for c in encabezado.columns
    ]

    tabla = valores.loc[es_dato]
    tabla.columns = etiquetas
    tabla.index = df.loc[es_dato, 0].astype(str).str.strip().str.rstrip("*")  # '*' = advertencia de muestreo
    tabla.index.name = "Pais"
    return tabla.loc[:, [e[-1] != "S.E." for e in etiquetas]]


def columna(tabla, *partes):
    # La única columna con un encabezado que empiece por cada una de las partes
    def coincide(etiqueta):
        return all(any(p.lower().startswith(parte.lower()) for p in etiqueta) for parte in partes)

    cols = [c for c in tabla.columns if coincide(c)]
    if len(cols) != 1:
        raise ValueError(f"{partes}: {len(cols)} columnas coinciden")
    return tabla[cols[0]]


def niveles(tabla, grupo):
    # % por nivel de un grupo ('All students', 'Girls' o 'Boys'), con códigos tipo 'Below1C', 'L1A', 'L2'
    out = {}
    for c in tabla.columns:
        if not c[0].startswith(grupo):
            continue
        for parte in c:
            m = re.match(r"(Below )?Level (\w+)", parte)
            if m:
                out[("Below" if m[1] else "L") + m[2].upper()] = tabla[c]
                break
    return pd.DataFrame(out)


def armonizar_niveles(niv, materia):
    # Deja los niveles con las mismas categorías que el Banco Mundial
    if materia == "Math":
        niv = niv.assign(Below1=niv["Below1C"] + niv["L1C"] + niv["L1B"], L1=niv["L1A"])
    etiquetas = list(NIVELES[MATERIA_COD[materia]].values())
    faltan = set(etiquetas) - set(niv.columns)
    if faltan:
        raise ValueError(f"{materia}: faltan los niveles {faltan}")
    return niv[etiquetas]

In [7]:
def construir_anio(anio):
    # Una fila por país (nombre de la OCDE) con las mismas columnas que el Banco Mundial
    f = FUENTES[anio]
    cols = {}
    for materia in MATERIAS.values():
        # Media y percentiles
        t = leer_tabla(f["media"][0], f["media"][1][materia])
        cols[f"PISA_{materia}_Total"] = columna(t, "Mean score")
        for p, etiqueta in PERCENTILES_OCDE.items():
            cols[f"PISA_{materia}_{p}"] = columna(t, "Percentiles", etiqueta)

        # Media por sexo
        t = leer_tabla(f["genero"][0], f["genero"][1][materia])
        for sexo, grupo in SEXOS_OCDE.items():
            cols[f"PISA_{materia}_{sexo}"] = columna(t, grupo, "Mean")

        # Niveles de competencia: total y por sexo
        t = leer_tabla(f["niveles"][0], f["niveles"][1][materia])
        for nivel, serie in armonizar_niveles(niveles(t, "All students"), materia).items():
            cols[f"PISA_{materia}_{nivel}_Total"] = serie

        t = leer_tabla(f["niveles_genero"][0], f["niveles_genero"][1][materia])
        for sexo, grupo in SEXOS_OCDE.items():
            for nivel, serie in armonizar_niveles(niveles(t, grupo), materia).items():
                cols[f"PISA_{materia}_{nivel}_{sexo}"] = serie

        # Origen migratorio (solo matemáticas y lectura, como en el Banco Mundial)
        if materia in f["migracion"][1]:
            t = leer_tabla(f["migracion"][0], f["migracion"][1][materia])
            pre = PREFIJO_MIGRACION[anio]
            cols[f"PISA_{materia}_MinProf_NonImmigrant"] = 100 - columna(t, *pre, "Non-immigrant")
            cols[f"PISA_{materia}_MinProf_Immigrant"] = 100 - columna(t, *pre, "Immigrant students", "All immigrant")

    df = pd.DataFrame(cols)
    df.insert(0, "Time", anio)
    return df


ocde = pd.concat([construir_anio(2022), construir_anio(2025)])
print(ocde.shape)
ocde["Time"].value_counts().sort_index()

(176, 101)


Time
2022    82
2025    94
Name: count, dtype: int64

## 3. Nombres de país → código ISO3

La OCDE usa nombres en inglés y el Banco Mundial y WPP2024 usan ISO3.

Hay participantes que no cubren todo el país. Se asignan al código del país, igual que hace el Banco Mundial con China (Shanghái en 2009–2012, B-S-J-G en 2015, B-S-J-Z en 2018):
- `B-S-J-Z (China)` → `CHN`, `Baku (Azerbaijan)` → `AZE`, `Dushanbe (Tajikistan)` → `TJK`, `Kurdistan Region (Iraq)` → `IRQ`, `Ukrainian regions` → `UKR`.

`OECD average` → `OED`, como en el Banco Mundial: su `OED` de 2018 (489,3 en matemáticas) es el promedio OCDE. `OECD total` y `OECD average-29` son otros agregados y se descartan.

In [8]:
NOMBRE_A_ISO3 = {
    "Albania": "ALB", "Argentina": "ARG", "Armenia": "ARM", "Australia": "AUS", "Austria": "AUT",
    "Azerbaijan": "AZE", "Baku (Azerbaijan)": "AZE", "B-S-J-Z (China)": "CHN", "Belgium": "BEL",
    "Brazil": "BRA", "Brunei Darussalam": "BRN", "Bulgaria": "BGR", "Cambodia": "KHM", "Canada": "CAN",
    "Chile": "CHL", "Chinese Taipei": "TWN", "Colombia": "COL", "Costa Rica": "CRI", "Croatia": "HRV",
    "Cyprus": "CYP", "Czech Republic": "CZE", "Czechia": "CZE", "Denmark": "DNK",
    "Dominican Republic": "DOM", "Dushanbe (Tajikistan)": "TJK", "Ecuador": "ECU", "El Salvador": "SLV",
    "Estonia": "EST", "Finland": "FIN", "France": "FRA", "Georgia": "GEO", "Germany": "DEU",
    "Greece": "GRC", "Guatemala": "GTM", "Hong Kong (China)": "HKG", "Hungary": "HUN", "Iceland": "ISL",
    "Indonesia": "IDN", "Ireland": "IRL", "Israel": "ISR", "Italy": "ITA", "Jamaica": "JAM",
    "Japan": "JPN", "Jordan": "JOR", "Kazakhstan": "KAZ", "Kenya": "KEN", "Korea": "KOR",
    "Kosovo": "XKX", "Kurdistan Region (Iraq)": "IRQ", "Kyrgyzstan": "KGZ", "Latvia": "LVA",
    "Lebanon": "LBN", "Lithuania": "LTU", "Luxembourg": "LUX", "Macao (China)": "MAC",
    "Malaysia": "MYS", "Malta": "MLT", "Mauritius": "MUS", "Mexico": "MEX", "Moldova": "MDA",
    "Mongolia": "MNG", "Montenegro": "MNE", "Morocco": "MAR", "Netherlands": "NLD",
    "New Zealand": "NZL", "North Macedonia": "MKD", "Norway": "NOR", "OECD average": "OED",
    "Palestinian Authority": "PSE", "Panama": "PAN", "Paraguay": "PRY", "Peru": "PER",
    "Philippines": "PHL", "Poland": "POL", "Portugal": "PRT", "Qatar": "QAT", "Romania": "ROU",
    "Rwanda": "RWA", "Saudi Arabia": "SAU", "Serbia": "SRB", "Singapore": "SGP",
    "Slovak Republic": "SVK", "Slovenia": "SVN", "Spain": "ESP", "Sweden": "SWE",
    "Switzerland": "CHE", "Thailand": "THA", "Türkiye": "TUR", "Ukrainian regions (17 of 27)": "UKR",
    "Ukrainian regions (18 of 27)": "UKR", "United Arab Emirates": "ARE", "United Kingdom": "GBR",
    "United States": "USA", "Uruguay": "URY", "Uzbekistan": "UZB", "Viet Nam": "VNM", "Zambia": "ZMB",
}
AGREGADOS_DESCARTADOS = {"OECD total", "OECD average-29"}

sin_codigo = set(ocde.index) - set(NOMBRE_A_ISO3) - AGREGADOS_DESCARTADOS
assert not sin_codigo, f"Nombres sin código ISO3: {sin_codigo}"

ocde = ocde.drop(index=list(AGREGADOS_DESCARTADOS), errors="ignore")
ocde.insert(0, "ISO3_code", ocde.index.map(NOMBRE_A_ISO3))
ocde = ocde.reset_index(drop=True)

assert not ocde.duplicated(["ISO3_code", "Time"]).any(), "Hay países repetidos en un mismo año"

## 4. Paridad de columnas y validación

Las dos fuentes deben tener exactamente las mismas columnas. Después se comparan con los datos de tendencia que trae el propio libro de 2025, que repite los valores de 2018 y 2022:
- **2018:** media del Banco Mundial frente a la de la OCDE (comprueba que las fuentes coinciden).
- **2018, origen migratorio:** la serie UIS del Banco Mundial frente a `100 − bajo rendimiento` de la OCDE (comprueba la conversión).
- **2022:** la media leída del libro de 2022 frente a la del libro de 2025 (comprueba la lectura).

In [9]:
solo_bm = set(banco_mundial.columns) - set(ocde.columns)
solo_ocde = set(ocde.columns) - set(banco_mundial.columns)
assert not solo_bm and not solo_ocde, f"Solo Banco Mundial: {solo_bm} | Solo OCDE: {solo_ocde}"
print(f"Paridad OK: {banco_mundial.shape[1]} columnas en ambas fuentes")

Paridad OK: 102 columnas en ambas fuentes


In [10]:
def tendencia(archivo, hoja, *partes):
    # Serie de un ciclo tomada de una tabla de tendencias de 2025, indexada por ISO3
    s = columna(leer_tabla(archivo, hoja), *partes)
    s = s[s.index.isin(NOMBRE_A_ISO3)]
    s.index = s.index.map(NOMBRE_A_ISO3)
    return s.groupby(level=0).first()


def comparar(nuestro, referencia, nombre):
    df = pd.concat({"nuestro": nuestro, "referencia": referencia}, axis=1).dropna()
    dif = (df["nuestro"] - df["referencia"]).abs()
    print(f"{nombre}: {len(df)} países | diferencia media {dif.mean():.2f} | máxima {dif.max():.2f} | r = {df.corr().iloc[0, 1]:.3f}")


ARCHIVO_TENDENCIAS = "PISA2025_cap2a_desempeno_niveles.xlsx"
ARCHIVO_MIGRACION_2025 = FUENTES[2025]["migracion"][0]

bm_2018 = banco_mundial[banco_mundial["Time"] == 2018].set_index("ISO3_code")
ocde_2022 = ocde[ocde["Time"] == 2022].set_index("ISO3_code")

comparar(bm_2018["PISA_Math_Total"],
         tendencia(ARCHIVO_TENDENCIAS, "Table I.B1.2a.38", "Mathematics performance", "PISA 2018"),
         "Media matemáticas 2018 (Banco Mundial vs OCDE)")
comparar(bm_2018["PISA_Math_MinProf_Immigrant"],
         100 - tendencia(ARCHIVO_MIGRACION_2025, "Table I.B1.2d.15", "PISA 2018", "Immigrant students", "All immigrant"),
         "MinProf inmigrantes matemáticas 2018 (UIS vs OCDE)")
comparar(ocde_2022["PISA_Math_Total"],
         tendencia(ARCHIVO_TENDENCIAS, "Table I.B1.2a.38", "Mathematics performance", "PISA 2022"),
         "Media matemáticas 2022 (libro 2022 vs libro 2025)")

Media matemáticas 2018 (Banco Mundial vs OCDE): 70 países | diferencia media 0.00 | máxima 0.00 | r = 1.000
MinProf inmigrantes matemáticas 2018 (UIS vs OCDE): 50 países | diferencia media 0.00 | máxima 0.00 | r = 1.000


Media matemáticas 2022 (libro 2022 vs libro 2025): 77 países | diferencia media 0.09 | máxima 6.36 | r = 1.000


## 5. Unión y variables derivadas

In [11]:
pisa = pd.concat([banco_mundial, ocde], ignore_index=True)

derivadas = {}
for materia in MATERIAS.values():
    # Brecha de género (Mujeres - Hombres)
    derivadas[f"Gap_{materia}_Fem_Male"] = pisa[f"PISA_{materia}_Female"] - pisa[f"PISA_{materia}_Male"]
    # Dispersión: distancia entre el percentil 90 y el 10
    derivadas[f"Spread_{materia}_P90_P10"] = pisa[f"PISA_{materia}_P90"] - pisa[f"PISA_{materia}_P10"]

pisa = pd.concat([pisa, pd.DataFrame(derivadas)], axis=1)
pisa = pisa.sort_values(["ISO3_code", "Time"]).reset_index(drop=True)
assert not pisa.duplicated(["ISO3_code", "Time"]).any()

print(pisa.shape)
pisa.groupby("Time")["ISO3_code"].nunique().rename("paises")

(593, 108)


Time
2000    42
2003    41
2006    56
2009    69
2012    62
2015    71
2018    78
2022    82
2025    92
Name: paises, dtype: int64

In [12]:
resumen(pisa).head(15)

,tipo,nulos,pct_nulos,unicos
PISA_Math_MinProf_Immigrant,float64,115,19.4,477
PISA_Science_Below1B_Male,float64,105,17.7,336
PISA_Science_L1B_Male,float64,105,17.7,350
PISA_Science_Below1B_Female,float64,105,17.7,332
PISA_Science_Below1B_Total,float64,105,17.7,335
PISA_Science_L1B_Total,float64,105,17.7,347
PISA_Science_L1B_Female,float64,105,17.7,349
PISA_Math_MinProf_NonImmigrant,float64,96,16.2,494
PISA_Reading_MinProf_Immigrant,float64,77,13.0,510
PISA_Reading_MinProf_NonImmigrant,float64,61,10.3,527


In [13]:
guardar_processed(pisa, "pisa_unificado.csv")

Guardado: data\processed\pisa_unificado.csv (593 filas)


WindowsPath('C:/Users/jomig/Desktop/Competencia-ASA/data/processed/pisa_unificado.csv')

## 6. Notas para el análisis

- **`OED`** es el promedio OCDE, no un país: hay que excluirlo antes de unir con WPP2024 o de promediar países.
- **Cobertura parcial:** `CHN` (solo algunas provincias), `AZE` (Bakú en 2022), `TJK` (Dusambé), `IRQ` (Kurdistán) y `UKR` (algunas regiones en 2022 y 2025) no representan a todo el país.
- **Origen migratorio:** en 2000–2018 viene de UIS y en 2022–2025 de la OCDE. La comparación de la sección 4 indica si son comparables.
- **Niveles parciales en ciencias:** 2000 no tiene niveles de ciencias y 2003 no tiene `Below1B` ni `L1B`, porque esas categorías no existían.
- **Revisiones de la OCDE:** el libro de 2025 corrige algunos valores de 2022. El más grande es Palestina (`PSE`), con 6,4 puntos de diferencia en matemáticas. Aquí se usa el valor publicado en el libro de 2022.
- **PISA 2022 y 2025** traen advertencias de muestreo (`*` en el nombre del país, que aquí se quitó). La lista está en la hoja `TOC` de cada libro.